# Locked Answer Generation and Evaluation

## Purpose

This notebook evaluates the frozen end-to-end RAG system on questions that were not used for retrieval or prompt development.

The system is already frozen:

- Hybrid TF-IDF and BGE retriever
- 140-word chunks with 25-word overlap
- Final retrieval Top K of 5
- Gemini 3.5 Flash generator
- `dev_v2` system prompt
- Temperature 0
- Thinking budget 0

## Evaluation rules

1. Do not change retrieval, prompting, model, or generation settings.
2. Generate every locked question using the same frozen pipeline.
3. Save each answer immediately so API limits or failures do not erase progress.
4. Preserve the first completed locked run.
5. Evaluate answer completeness, citation support, faithfulness, forbidden inferences, and refusal quality.
6. Do not tune the system using locked results and report the tuned result as the same unbiased evaluation.

The free API quota may require generation across multiple days. Checkpointing is part of the evaluation design, not an optional convenience.

In [ ]:
import json
from pathlib import Path


def find_project_root(start_path):
    for candidate in [
        start_path,
        *start_path.parents,
    ]:
        if (
            candidate.joinpath(
                "pyproject.toml"
            ).exists()
            and candidate.joinpath(
                "evaluation"
            ).exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root."
    )


PROJECT_ROOT = find_project_root(
    Path.cwd().resolve()
)

generation_selection_path = (
    PROJECT_ROOT
    / "evaluation"
    / "generation_selection.json"
)

generation_selection = json.loads(
    generation_selection_path.read_text(
        encoding="utf-8"
    )
)

selected_config_path = (
    PROJECT_ROOT
    / "evaluation"
    / generation_selection[
        "generation_config_file"
    ]
)

selected_generation_config = json.loads(
    selected_config_path.read_text(
        encoding="utf-8"
    )
)

assert (
    generation_selection["status"]
    == "frozen_before_locked_answer_evaluation"
)

assert (
    generation_selection[
        "selected_prompt_version"
    ]
    == "dev_v2"
)

assert (
    selected_generation_config[
        "system_prompt_sha256"
    ]
    == generation_selection[
        "selected_prompt_sha256"
    ]
)

print("Project root:", PROJECT_ROOT)
print(
    "Generation status:",
    generation_selection["status"],
)
print(
    "Selected prompt:",
    generation_selection[
        "selected_prompt_version"
    ],
)
print(
    "Selected model:",
    generation_selection[
        "selected_model"
    ],
)

verify that the locked dataset has not changed

In [ ]:
import hashlib


locked_questions_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_questions.json"
)

locked_hash_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_questions.sha256"
)

locked_questions_bytes = (
    locked_questions_path.read_bytes()
)

computed_locked_hash = hashlib.sha256(
    locked_questions_bytes
).hexdigest()

stored_locked_hash = (
    locked_hash_path
    .read_text(encoding="utf-8")
    .strip()
    .split()[0]
)

assert computed_locked_hash == stored_locked_hash

locked_questions = json.loads(
    locked_questions_bytes.decode("utf-8")
)

locked_question_ids = [
    item["question_id"]
    for item in locked_questions
]

assert len(locked_questions) == 28
assert len(set(locked_question_ids)) == 28
assert all(
    item["split"] == "locked"
    for item in locked_questions
)

answerable_count = sum(
    item["answerable"]
    for item in locked_questions
)

refusal_count = (
    len(locked_questions)
    - answerable_count
)

print("Locked questions:", len(locked_questions))
print("Answerable:", answerable_count)
print("Refusals:", refusal_count)
print("Verified SHA-256:", computed_locked_hash)

In [ ]:
import pandas as pd
from sentence_transformers import (
    SentenceTransformer,
)

from aviation_rag.retrieval import (
    HybridRetriever,
    SemanticRetriever,
    TfidfRetriever,
)


retrieval_config_path = (
    PROJECT_ROOT
    / "evaluation"
    / "retrieval_config.json"
)

retrieval_config = json.loads(
    retrieval_config_path.read_text(
        encoding="utf-8"
    )
)

chunks_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "chunks.jsonl"
)

chunks = pd.read_json(
    chunks_path,
    lines=True,
)

assert len(chunks) == 368
assert chunks["chunk_id"].is_unique

semantic_settings = (
    retrieval_config["semantic_retriever"]
)

hybrid_settings = (
    retrieval_config["hybrid_retriever"]
)

semantic_model = SentenceTransformer(
    semantic_settings["model"]
)

lexical_retriever = TfidfRetriever(
    chunks=chunks
)

semantic_retriever = SemanticRetriever(
    chunks=chunks,
    model=semantic_model,
    query_prefix=(
        semantic_settings[
            "query_instruction"
        ]
    ),
)

hybrid_retriever = HybridRetriever(
    lexical_retriever=lexical_retriever,
    semantic_retriever=semantic_retriever,
    candidate_k=(
        hybrid_settings["candidate_k"]
    ),
    rrf_constant=(
        hybrid_settings["rrf_constant"]
    ),
    lexical_weight=(
        hybrid_settings["lexical_weight"]
    ),
)

print("Chunks:", len(chunks))
print(
    "Semantic model:",
    semantic_settings["model"],
)
print(
    "Embedding matrix:",
    semantic_retriever
    .chunk_embeddings
    .shape,
)
print(
    "Candidate K:",
    hybrid_retriever.candidate_k,
)
print(
    "Final retrieval K:",
    selected_generation_config[
        "retrieval_top_k"
    ],
)

In [ ]:
import os
import re
import time

from dotenv import load_dotenv
from google import genai
from google.genai import types


load_dotenv(
    PROJECT_ROOT / ".env"
)

gemini_api_key = os.getenv(
    "GEMINI_API_KEY"
)

if not gemini_api_key:
    raise RuntimeError(
        "GEMINI_API_KEY was not found in .env"
    )

gemini_client = genai.Client(
    api_key=gemini_api_key
)

SYSTEM_INSTRUCTIONS = (
    selected_generation_config[
        "system_instructions"
    ]
)

GENERATION_MODEL = (
    selected_generation_config["model"]
)

GENERATION_TEMPERATURE = (
    selected_generation_config[
        "temperature"
    ]
)

MAX_OUTPUT_TOKENS = (
    selected_generation_config[
        "max_output_tokens"
    ]
)

THINKING_BUDGET = (
    selected_generation_config[
        "thinking_budget"
    ]
)

RETRIEVAL_TOP_K = (
    selected_generation_config[
        "retrieval_top_k"
    ]
)

assert (
    hashlib.sha256(
        SYSTEM_INSTRUCTIONS.encode("utf-8")
    ).hexdigest()
    == generation_selection[
        "selected_prompt_sha256"
    ]
)

print("Model:", GENERATION_MODEL)
print(
    "Prompt version:",
    generation_selection[
        "selected_prompt_version"
    ],
)
print(
    "Temperature:",
    GENERATION_TEMPERATURE,
)
print(
    "Thinking budget:",
    THINKING_BUDGET,
)
print(
    "Retrieval Top K:",
    RETRIEVAL_TOP_K,
)

In [ ]:
def build_retrieved_context(
    question,
    retriever,
    top_k,
):
    retrieved = retriever.retrieve(
        query=question,
        top_k=top_k,
    )

    source_blocks = []

    for source_number, row in enumerate(
        retrieved.itertuples(index=False),
        start=1,
    ):
        source_blocks.append(
            "\n".join(
                [
                    f"[SOURCE {source_number}]",
                    (
                        "Citation: "
                        f"[{row.document_id}, "
                        f"p. {int(row.page_number)}]"
                    ),
                    "Text:",
                    row.chunk_text,
                ]
            )
        )

    context = "\n\n".join(
        source_blocks
    )

    return context, retrieved


def build_answer_messages(
    question,
    context,
):
    user_message = f"""
Question:
{question}

Evidence:
{context}

Provide a grounded answer with page citations.
""".strip()

    return [
        {
            "role": "system",
            "content": SYSTEM_INSTRUCTIONS,
        },
        {
            "role": "user",
            "content": user_message,
        },
    ]


print("Context and message builders ready.")

In [ ]:
CITATION_PATTERN = re.compile(
    r"\[([A-Za-z0-9_-]+),\s*p\.\s*(\d+)\]"
)


def validate_citations(
    answer,
    retrieved,
):
    citation_matches = (
        CITATION_PATTERN.findall(answer)
    )

    cited_sources = sorted(
        {
            (
                document_id,
                int(page_number),
            )
            for document_id, page_number
            in citation_matches
        }
    )

    valid_sources = {
        (
            row.document_id,
            int(row.page_number),
        )
        for row in retrieved.itertuples(
            index=False
        )
    }

    invalid_citations = [
        source
        for source in cited_sources
        if source not in valid_sources
    ]

    return {
        "citation_count": len(
            cited_sources
        ),
        "cited_sources": cited_sources,
        "invalid_citations": (
            invalid_citations
        ),
        "has_citation": bool(
            cited_sources
        ),
        "all_citations_valid": (
            bool(cited_sources)
            and not invalid_citations
        ),
    }


def generate_grounded_answer(
    question,
):
    context, retrieved = (
        build_retrieved_context(
            question=question,
            retriever=hybrid_retriever,
            top_k=RETRIEVAL_TOP_K,
        )
    )

    messages = build_answer_messages(
        question=question,
        context=context,
    )

    response = (
        gemini_client.models.generate_content(
            model=GENERATION_MODEL,
            contents=messages[1]["content"],
            config=types.GenerateContentConfig(
                system_instruction=(
                    messages[0]["content"]
                ),
                temperature=(
                    GENERATION_TEMPERATURE
                ),
                max_output_tokens=(
                    MAX_OUTPUT_TOKENS
                ),
                thinking_config=(
                    types.ThinkingConfig(
                        thinking_budget=(
                            THINKING_BUDGET
                        ),
                    )
                ),
            ),
        )
    )

    if not response.text:
        raise RuntimeError(
            "Gemini returned no answer text."
        )

    answer = response.text.strip()

    return {
        "answer": answer,
        "context": context,
        "retrieved": retrieved,
        "citation_check": (
            validate_citations(
                answer=answer,
                retrieved=retrieved,
            )
        ),
        "finish_reason": str(
            response.candidates[
                0
            ].finish_reason
        ),
        "usage_metadata": (
            response.usage_metadata
        ),
    }


print("Citation validator ready.")
print("Frozen generation function ready.")

In [ ]:
TRANSIENT_ERROR_MARKERS = (
    "429",
    "500",
    "502",
    "503",
    "504",
    "UNAVAILABLE",
    "RESOURCE_EXHAUSTED",
)

DAILY_QUOTA_MARKERS = (
    "GenerateRequestsPerDay",
    "requests per day",
    "free_tier_requests",
)


def generate_with_retry(
    question,
    max_attempts=4,
):
    for attempt in range(
        1,
        max_attempts + 1,
    ):
        try:
            return generate_grounded_answer(
                question=question
            )

        except Exception as error:
            error_text = str(error)

            daily_quota_exhausted = any(
                marker in error_text
                for marker
                in DAILY_QUOTA_MARKERS
            )

            if daily_quota_exhausted:
                raise

            is_transient = any(
                marker in error_text
                for marker
                in TRANSIENT_ERROR_MARKERS
            )

            if (
                not is_transient
                or attempt == max_attempts
            ):
                raise

            wait_seconds = 15 * (
                2 ** (attempt - 1)
            )

            print(
                "Temporary API failure. "
                f"Retrying in {wait_seconds}s "
                f"({attempt}/{max_attempts})."
            )

            time.sleep(wait_seconds)


print("Retry controller ready.")

In [ ]:
from datetime import datetime, timezone


locked_output_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_answers_first_run.jsonl"
)

locked_records = []

if locked_output_path.exists():
    locked_records = [
        json.loads(line)
        for line in locked_output_path
        .read_text(encoding="utf-8")
        .splitlines()
        if line.strip()
    ]


for record in locked_records:
    assert (
        record["locked_dataset_sha256"]
        == computed_locked_hash
    )
    assert (
        record["system_prompt_sha256"]
        == generation_selection[
            "selected_prompt_sha256"
        ]
    )
    assert (
        record["model"]
        == GENERATION_MODEL
    )


completed_ids = {
    record["question_id"]
    for record in locked_records
}

assert len(completed_ids) == len(
    locked_records
)

assert completed_ids.issubset(
    set(locked_question_ids)
)


def save_locked_records():
    content = "".join(
        json.dumps(
            record,
            ensure_ascii=False,
        )
        + "\n"
        for record in locked_records
    )

    locked_output_path.write_bytes(
        content.encode("utf-8")
    )


for item in locked_questions:
    question_id = item["question_id"]

    if question_id in completed_ids:
        print(
            "Skipping completed:",
            question_id,
        )
        continue

    print("Generating:", question_id)

    try:
        result = generate_with_retry(
            question=item["question"]
        )

    except Exception as error:
        print(
            "Stopped at:",
            question_id,
            type(error).__name__,
            str(error),
        )
        break

    usage = result["usage_metadata"]

    retrieved_chunks = []

    for rank, row in enumerate(
        result["retrieved"].itertuples(
            index=False
        ),
        start=1,
    ):
        retrieved_chunks.append(
            {
                "rank": rank,
                "chunk_id": row.chunk_id,
                "document_id": (
                    row.document_id
                ),
                "page_number": int(
                    row.page_number
                ),
                "score": float(row.score),
                "chunk_text": row.chunk_text,
            }
        )

    locked_records.append(
        {
            "question_id": question_id,
            "question": item["question"],
            "category": item["category"],
            "answerable": item["answerable"],
            "answer": result["answer"],
            "citation_check": (
                result["citation_check"]
            ),
            "finish_reason": (
                result["finish_reason"]
            ),
            "retrieved_chunks": (
                retrieved_chunks
            ),
            "retrieved_context_sha256": (
                hashlib.sha256(
                    result["context"].encode(
                        "utf-8"
                    )
                ).hexdigest()
            ),
            "locked_dataset_sha256": (
                computed_locked_hash
            ),
            "prompt_version": (
                generation_selection[
                    "selected_prompt_version"
                ]
            ),
            "system_prompt_sha256": (
                generation_selection[
                    "selected_prompt_sha256"
                ]
            ),
            "model": GENERATION_MODEL,
            "temperature": (
                GENERATION_TEMPERATURE
            ),
            "thinking_budget": (
                THINKING_BUDGET
            ),
            "retrieval_top_k": (
                RETRIEVAL_TOP_K
            ),
            "prompt_tokens": getattr(
                usage,
                "prompt_token_count",
                None,
            ),
            "output_tokens": getattr(
                usage,
                "candidates_token_count",
                None,
            ),
            "total_tokens": getattr(
                usage,
                "total_token_count",
                None,
            ),
            "generated_at_utc": (
                datetime.now(
                    timezone.utc
                ).isoformat()
            ),
        }
    )

    save_locked_records()
    completed_ids.add(question_id)

    time.sleep(15)


print(
    "Completed locked answers:",
    len(locked_records),
    "/",
    len(locked_questions),
)

print("Saved:", locked_output_path)

Next, freeze the generated answers before manual evaluation.

In [ ]:
from collections import Counter


locked_records = [
    json.loads(line)
    for line in locked_output_path
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

locked_record_ids = {
    record["question_id"]
    for record in locked_records
}

assert len(locked_records) == 28
assert locked_record_ids == set(
    locked_question_ids
)

assert all(
    record["locked_dataset_sha256"]
    == computed_locked_hash
    for record in locked_records
)

assert all(
    record["system_prompt_sha256"]
    == generation_selection[
        "selected_prompt_sha256"
    ]
    for record in locked_records
)

assert all(
    record["model"]
    == GENERATION_MODEL
    for record in locked_records
)

assert all(
    len(record["retrieved_chunks"])
    == RETRIEVAL_TOP_K
    for record in locked_records
)

assert all(
    record["retrieved_context_sha256"]
    for record in locked_records
)

print("Locked answer run passed validation.")
print("Answers:", len(locked_records))

print(
    "Finish reasons:",
    Counter(
        record["finish_reason"]
        for record in locked_records
    ),
)

print(
    "Total prompt tokens:",
    sum(
        record["prompt_tokens"] or 0
        for record in locked_records
    ),
)

print(
    "Total output tokens:",
    sum(
        record["output_tokens"] or 0
        for record in locked_records
    ),
)

## Locked Generation Run Validation

The first locked answer-generation run completed successfully for all 28 questions.

All responses ended with `FinishReason.STOP`. This means Gemini completed each response normally instead of reaching the output-token limit or stopping because of an API failure. It does not prove that the answers are correct, complete, or faithful.

The run used:

- 43,456 prompt tokens
- 3,541 output tokens

Prompt tokens are much higher because every request includes the system instructions, user question, and five retrieved evidence chunks. Output tokens contain only the generated answers.

The run required multiple days because of the free API request limit. Checkpointing preserved each completed answer and allowed the batch to resume without regenerating earlier questions.

This validation confirms technical completeness and configuration consistency. Answer quality must still be evaluated manually.

In [ ]:
locked_answer_bytes = (
    locked_output_path.read_bytes()
)

locked_answers_sha256 = (
    hashlib.sha256(
        locked_answer_bytes
    ).hexdigest()
)

generated_timestamps = [
    record["generated_at_utc"]
    for record in locked_records
]

locked_run_metadata = {
    "status": "generation_completed",
    "locked_dataset_sha256": (
        computed_locked_hash
    ),
    "locked_answers_sha256": (
        locked_answers_sha256
    ),
    "question_count": len(
        locked_records
    ),
    "answerable_questions": sum(
        record["answerable"]
        for record in locked_records
    ),
    "refusal_questions": sum(
        not record["answerable"]
        for record in locked_records
    ),
    "prompt_version": (
        generation_selection[
            "selected_prompt_version"
        ]
    ),
    "system_prompt_sha256": (
        generation_selection[
            "selected_prompt_sha256"
        ]
    ),
    "model": GENERATION_MODEL,
    "temperature": (
        GENERATION_TEMPERATURE
    ),
    "thinking_budget": (
        THINKING_BUDGET
    ),
    "retrieval_top_k": (
        RETRIEVAL_TOP_K
    ),
    "run_started_at_utc": min(
        generated_timestamps
    ),
    "run_completed_at_utc": max(
        generated_timestamps
    ),
    "prompt_tokens": sum(
        record["prompt_tokens"] or 0
        for record in locked_records
    ),
    "output_tokens": sum(
        record["output_tokens"] or 0
        for record in locked_records
    ),
    "total_tokens": sum(
        record["total_tokens"] or 0
        for record in locked_records
    ),
    "finish_reasons": dict(
        Counter(
            record["finish_reason"]
            for record in locked_records
        )
    ),
    "answers_file": (
        locked_output_path.name
    ),
    "manual_evaluation_status": (
        "not_started"
    ),
}

locked_run_metadata_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_answer_run.json"
)

if locked_run_metadata_path.exists():
    existing_locked_run = json.loads(
        locked_run_metadata_path.read_text(
            encoding="utf-8"
        )
    )
    assert (
        existing_locked_run
        == locked_run_metadata
    )
else:
    locked_run_metadata_path.write_bytes(
        (
            json.dumps(
                locked_run_metadata,
                indent=2,
                ensure_ascii=False,
            )
            + "\n"
        ).encode("utf-8")
    )


current_selection = json.loads(
    generation_selection_path.read_text(
        encoding="utf-8"
    )
)

current_selection[
    "locked_answer_evaluation_status"
] = "generation_completed_evaluation_pending"

generation_selection_path.write_bytes(
    (
        json.dumps(
            current_selection,
            indent=2,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")
)

print(
    "Locked answers SHA-256:",
    locked_answers_sha256,
)
print(
    "Run status:",
    locked_run_metadata["status"],
)
print(
    "Manual evaluation:",
    locked_run_metadata[
        "manual_evaluation_status"
    ],
)
print(
    "Saved metadata:",
    locked_run_metadata_path,
)

In [ ]:
locked_question_by_id = {
    item["question_id"]: item
    for item in locked_questions
}

locked_answer_by_id = {
    record["question_id"]: record
    for record in locked_records
}

locked_scores_path = (
    PROJECT_ROOT
    / "evaluation"
    / "locked_answer_scores_first_run.jsonl"
)

locked_scores = []

if locked_scores_path.exists():
    locked_scores = [
        json.loads(line)
        for line in locked_scores_path
        .read_text(encoding="utf-8")
        .splitlines()
        if line.strip()
    ]


def display_locked_case(question_id):
    question = locked_question_by_id[
        question_id
    ]
    answer = locked_answer_by_id[
        question_id
    ]

    print("Question:")
    print(question["question"])

    print("\nExpected answer:")
    print(question["expected_answer"])

    print("\nRequired facts:")
    for index, fact in enumerate(
        question["required_facts"],
        start=1,
    ):
        print(f"{index}. {fact}")

    print("\nForbidden inferences:")
    if question["forbidden_inferences"]:
        for inference in question[
            "forbidden_inferences"
        ]:
            print("-", inference)
    else:
        print("None")

    print("\nGenerated answer:")
    print(answer["answer"])

    print("\nCitation check:")
    print(answer["citation_check"])


print(
    "Existing locked scores:",
    len(locked_scores),
)

display_locked_case("LOCK_REG_001")

In [ ]:
def save_locked_score(score):
    global locked_scores

    locked_scores = [
        existing
        for existing in locked_scores
        if existing["question_id"]
        != score["question_id"]
    ]

    locked_scores.append(score)

    content = "".join(
        json.dumps(
            item,
            ensure_ascii=False,
        )
        + "\n"
        for item in locked_scores
    )

    locked_scores_path.write_bytes(
        content.encode("utf-8")
    )


def record_locked_answerable_score(
    question_id,
    required_facts_met,
    citation_support_pass,
    unsupported_claim_count,
    forbidden_inference_pass,
    notes,
):
    question = locked_question_by_id[
        question_id
    ]
    answer = locked_answer_by_id[
        question_id
    ]

    required_facts_total = len(
        question["required_facts"]
    )

    score = {
        "question_id": question_id,
        "category": question["category"],
        "answerable": True,
        "required_facts_met": (
            required_facts_met
        ),
        "required_facts_total": (
            required_facts_total
        ),
        "required_fact_coverage": (
            required_facts_met
            / required_facts_total
        ),
        "citation_validity_pass": (
            answer["citation_check"][
                "all_citations_valid"
            ]
        ),
        "citation_support_pass": (
            citation_support_pass
        ),
        "unsupported_claim_count": (
            unsupported_claim_count
        ),
        "faithfulness_pass": (
            unsupported_claim_count == 0
        ),
        "forbidden_inference_pass": (
            forbidden_inference_pass
        ),
        "refusal_correct": None,
        "notes": notes,
    }

    save_locked_score(score)


record_locked_answerable_score(
    question_id="LOCK_REG_001",
    required_facts_met=1,
    citation_support_pass=True,
    unsupported_claim_count=0,
    forbidden_inference_pass=True,
    notes=(
        "Correctly states the required first action "
        "and cites principal Code page 50."
    ),
)

display_locked_case("LOCK_REG_002")

In [ ]:
record_locked_answerable_score(
    question_id="LOCK_REG_002",
    required_facts_met=3,
    citation_support_pass=True,
    unsupported_claim_count=0,
    forbidden_inference_pass=True,
    notes=(
        "Correctly states both baggage complaint "
        "deadlines and the consequence of missing them."
    ),
)

display_locked_case("LOCK_REG_003")

The remaining locked answers were scored using AI-assisted rubric review, with question-level scores and notes preserved in locked_answer_scores_first_run.jsonl.

## Completed Manual Evaluation

All 28 locked answers were reviewed against the frozen expected answers, required facts, forbidden inferences, generated claims, citations, and retrieved evidence.

The first two questions above remain as worked examples of manual scoring. The completed question-level scores are stored in `evaluation/locked_answer_scores_first_run.jsonl` and loaded below.

Two limitations were found during review:

- Two answers used malformed multi-page citation syntax that the automatic validator did not detect.
- The five locked refusal questions contain no structured `required_facts`, so refusal completeness is evaluated against their frozen expected behaviour instead of refusal fact coverage.

In [ ]:
locked_scores = [
    json.loads(line)
    for line in locked_scores_path
    .read_text(encoding="utf-8")
    .splitlines()
    if line.strip()
]

assert len(locked_scores) == 28
assert {
    score["question_id"]
    for score in locked_scores
} == set(locked_question_ids)

locked_scores_df = pd.DataFrame(
    locked_scores
)

answerable_scores = locked_scores_df.loc[
    locked_scores_df["answerable"]
].copy()

refusal_scores = locked_scores_df.loc[
    ~locked_scores_df["answerable"]
].copy()

answerable_scores["full_pass"] = (
    answerable_scores[
        "required_fact_coverage"
    ].eq(1.0)
    & answerable_scores[
        "citation_validity_pass"
    ].eq(True)
    & answerable_scores[
        "citation_support_pass"
    ].eq(True)
    & answerable_scores[
        "faithfulness_pass"
    ].eq(True)
    & answerable_scores[
        "forbidden_inference_pass"
    ].eq(True)
)

refusal_scores["clean_refusal_pass"] = (
    refusal_scores[
        "refusal_correct"
    ].eq(True)
    & refusal_scores[
        "refusal_expected_behavior_pass"
    ].eq(True)
    & refusal_scores[
        "unsupported_claim_count"
    ].eq(0)
    & refusal_scores[
        "forbidden_inference_pass"
    ].eq(True)
)

summary = pd.Series(
    {
        "answerable_questions": len(
            answerable_scores
        ),
        "answerable_micro_fact_coverage": (
            answerable_scores[
                "required_facts_met"
            ].sum()
            / answerable_scores[
                "required_facts_total"
            ].sum()
        ),
        "answerable_fact_complete_rate": (
            answerable_scores[
                "required_fact_coverage"
            ].eq(1.0).mean()
        ),
        "answerable_full_pass_rate": (
            answerable_scores[
                "full_pass"
            ].mean()
        ),
        "answerable_citation_validity_rate": (
            answerable_scores[
                "citation_validity_pass"
            ].mean()
        ),
        "answerable_citation_support_rate": (
            answerable_scores[
                "citation_support_pass"
            ].mean()
        ),
        "answerable_faithfulness_rate": (
            answerable_scores[
                "faithfulness_pass"
            ].mean()
        ),
        "refusal_questions": len(
            refusal_scores
        ),
        "correct_refusal_rate": (
            refusal_scores[
                "refusal_correct"
            ].mean()
        ),
        "clean_refusal_pass_rate": (
            refusal_scores[
                "clean_refusal_pass"
            ].mean()
        ),
    }
)

print(summary.round(3))

### Overall Result Interpretation

The locked system recovered 76.8% of all required answer facts. However, only 69.6% of answerable questions contained every required fact, and the stricter full-pass rate fell to 60.9% after citation and safety requirements were included.

Faithfulness remained 100%, meaning the model avoided unsupported factual claims in answerable responses. This does not prove success because faithful answers can still omit evidence or refuse when retrieval fails.

Citation support reached 95.7%, but citation validity was lower at 87.0%. Manual review caught malformed multi-page citations that the regular-expression validator ignored. This exposes a weakness in the validator, not only in the generator.

All five refusal decisions were correct, but only 60% met the complete frozen expected behaviour. Two live-status refusals correctly refused but omitted expected reference-number guidance. This also exposes tension between the frozen prompt, which prohibited reference-number instructions for live requests, and the locked expected answers.

In [ ]:
category_summary = (
    answerable_scores
    .groupby("category")
    .agg(
        question_count=(
            "question_id",
            "count",
        ),
        facts_met=(
            "required_facts_met",
            "sum",
        ),
        facts_total=(
            "required_facts_total",
            "sum",
        ),
        mean_question_fact_coverage=(
            "required_fact_coverage",
            "mean",
        ),
        full_pass_rate=(
            "full_pass",
            "mean",
        ),
    )
)

category_summary[
    "micro_fact_coverage"
] = (
    category_summary["facts_met"]
    / category_summary["facts_total"]
)

category_summary = category_summary[
    [
        "question_count",
        "micro_fact_coverage",
        "mean_question_fact_coverage",
        "full_pass_rate",
    ]
]

print("Category summary:")
print(category_summary.round(3))

answer_failures = answerable_scores.loc[
    ~answerable_scores["full_pass"],
    [
        "question_id",
        "category",
        "required_fact_coverage",
        "citation_validity_pass",
        "citation_support_pass",
        "notes",
    ],
]

print("Answerable questions without a full pass:")
print(answer_failures)

print("Refusal review:")
print(
    refusal_scores[
        [
            "question_id",
            "refusal_correct",
            "refusal_expected_behavior_pass",
            "clean_refusal_pass",
            "notes",
        ]
    ]
)

### Category and Failure Interpretation

Direct regulatory lookup was strongest, with 100% fact coverage and a 100% full-pass rate. Report fact retrieval also achieved 100% fact coverage, but malformed multi-page citation syntax reduced its full-pass rate.

Temporal reasoning remained mostly effective, but one question failed because the commencement page was not retrieved. Amendment comparison was weak because several questions required evidence from multiple legal versions or pages. Cross-document synthesis failed completely, matching the earlier locked retrieval diagnosis.

The results confirm that the main weakness is not hallucination. It is incomplete evidence selection for temporal, amendment, and synthesis questions. Prompt changes cannot recover evidence that retrieval did not supply.

In [ ]:
locked_scores_sha256 = hashlib.sha256(
    locked_scores_path.read_bytes()
).hexdigest()

locked_run_metadata = json.loads(
    locked_run_metadata_path.read_text(
        encoding="utf-8"
    )
)

locked_run_metadata[
    "manual_evaluation_status"
] = "completed"

locked_run_metadata[
    "locked_scores_sha256"
] = locked_scores_sha256

locked_run_metadata[
    "manual_evaluation_summary"
] = {
    key: float(value)
    for key, value in summary.items()
}

locked_run_metadata_path.write_bytes(
    (
        json.dumps(
            locked_run_metadata,
            indent=2,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")
)

current_selection = json.loads(
    generation_selection_path.read_text(
        encoding="utf-8"
    )
)

current_selection[
    "locked_answer_evaluation_status"
] = "completed"

current_selection[
    "locked_answer_run_metadata"
] = locked_run_metadata_path.name

generation_selection_path.write_bytes(
    (
        json.dumps(
            current_selection,
            indent=2,
            ensure_ascii=False,
        )
        + "\n"
    ).encode("utf-8")
)

print("Locked scores SHA-256:", locked_scores_sha256)
print("Manual evaluation: completed")
print("Locked answer evaluation: completed")

## Phase Completion

The first locked end-to-end answer evaluation is now preserved. The frozen system achieved strong direct lookup and report fact retrieval, but amendment comparison and cross-document synthesis remained unreliable.

These locked results must not be used to tune the current system and then be reported as the same unbiased evaluation. Any improved retriever or prompt must be developed on development data or a new development set and evaluated on a new untouched holdout set.